In [1]:
# Python Standard Library
import sys

# Community Modules
import numpy as np
from scipy import stats
import pandas as pd
from pandas.testing import assert_frame_equal

# My Modules
sys.path.insert(0, "../code")
import dataset_utils as du

In [2]:
rng = np.random.default_rng(1415)
train_frac = 0.50

In [3]:
file_dataset = "../data/forSNR/dataset.parquet"
file_signal_only = "../data/forSNR/signal.parquet"
file_noise_only = "../data/forSNR/noise.parquet"

df_dataset = pd.read_parquet(file_dataset)
df_signal = pd.read_parquet(file_signal_only)
df_noise = pd.read_parquet(file_noise_only)

wvl, spectra, df_meta = unpack_dataset(df_dataset)
wvl_s, spectra_signal, df_meta_signal = unpack_dataset(df_signal)
wvl_n, spectra_noise, df_meta_noise = unpack_dataset(df_noise)

assert np.all(wvl == wvl_s)
assert np.all(wvl == wvl_n)
del wvl_s, wvl_n

assert_frame_equal(df_meta, df_meta_signal)
assert_frame_equal(df_meta_signal, df_meta_noise)

In [4]:
df_dataset["Training Set"] = False
df_signal["Training Set"] = False
df_noise["Training Set"] = False

classes = df_dataset["SN Subtype ID"].unique()
for subtype in classes:
    subtype_df = df_meta[df_meta["SN Subtype ID"] == subtype]
    
    subtype_SNe = subtype_df["SN Name"].unique()
    rng.shuffle(subtype_SNe)

    subtype_num_SNe = subtype_SNe.size
    train_size = int(np.ceil(subtype_num_SNe * train_frac))

    subtype_SNe_trn = subtype_SNe[:train_size]
    subtype_SNe_tst = subtype_SNe[train_size:]

    mask_trn = subtype_df["SN Name"].isin(subtype_SNe_trn)
    mask_tst = ~mask_trn

    ind_trn = subtype_df.index[mask_trn]
    ind_tst = subtype_df.index[mask_tst]

    df_dataset.loc[ind_trn, "Training Set"] = True
    df_dataset.loc[ind_tst, "Training Set"] = False

    df_signal.loc[ind_trn, "Training Set"] = True
    df_signal.loc[ind_tst, "Training Set"] = False

    df_noise.loc[ind_trn, "Training Set"] = True
    df_noise.loc[ind_tst, "Training Set"] = False

assert np.all(df_dataset["Training Set"] == df_signal["Training Set"])
assert np.all(df_dataset["Training Set"] == df_noise["Training Set"])

In [5]:
df_dataset["Training Set"].sum(), (~df_dataset["Training Set"]).sum()

(1692, 1882)

In [6]:
def get_table_train_test_split_by_spectra(x):
    num_trn = x["Training Set"].sum()
    num_tst = (~x["Training Set"]).sum()
    
    return x.shape[0], num_trn, num_tst

df_dataset.groupby(by=["SN Subtype"])[["SN Subtype", "Training Set"]].apply(get_table_train_test_split_by_spectra)

SN Subtype
IIP             (100, 52, 48)
IIb            (205, 84, 121)
Ia-91T        (339, 197, 142)
Ia-91bg        (222, 99, 123)
Ia-norm     (2034, 908, 1126)
Iax              (60, 29, 31)
Ib-norm        (198, 118, 80)
Ibn               (26, 22, 4)
Ic-broad       (210, 81, 129)
Ic-norm        (180, 102, 78)
dtype: object

In [7]:
def get_table_train_test_split_by_SN(x):
    SNe = x["SN Name"].unique()
    num_SNe = SNe.size

    num_SNe_trn = 0
    num_SNe_tst = 0
    for SN in SNe:
        SN_mask = x["SN Name"] == SN

        # If any of the spectra associated with this SN are in the training set...
        if np.any(x.loc[SN_mask, "Training Set"]):
            # Assert that all of them are.
            assert np.all(x.loc[x["SN Name"] == SN, "Training Set"])

        # If any of the spectra associated with this SN are in the testing set...
        elif np.any(~x.loc[SN_mask, "Training Set"]):
            # Assert that all of them are.
            assert not np.all(x.loc[SN_mask, "Training Set"])

        if np.all(x.loc[SN_mask, "Training Set"]):
            num_SNe_trn += 1
        else:
            num_SNe_tst += 1
            
    return num_SNe, num_SNe_trn, num_SNe_tst

df_dataset.groupby(by=["SN Subtype"])[["SN Subtype", "SN Name", "Training Set"]].apply(get_table_train_test_split_by_SN)

SN Subtype
IIP               (6, 3, 3)
IIb             (19, 10, 9)
Ia-91T         (27, 14, 13)
Ia-91bg        (28, 14, 14)
Ia-norm     (239, 120, 119)
Iax               (3, 2, 1)
Ib-norm        (22, 11, 11)
Ibn               (3, 2, 1)
Ic-broad       (20, 10, 10)
Ic-norm         (19, 10, 9)
dtype: object

In [8]:
df_dataset_trn = df_dataset[df_dataset["Training Set"]].drop(columns="Training Set")
df_dataset_tst = df_dataset[~df_dataset["Training Set"]].drop(columns="Training Set")

df_signal_trn = df_signal[df_signal["Training Set"]].drop(columns="Training Set")
df_signal_tst = df_signal[~df_signal["Training Set"]].drop(columns="Training Set")

df_noise_trn = df_noise[df_noise["Training Set"]].drop(columns="Training Set")
df_noise_tst = df_noise[~df_noise["Training Set"]].drop(columns="Training Set")

In [9]:
df_dataset_trn.shape, df_signal_trn.shape, df_noise_trn.shape

((1692, 1044), (1692, 1044), (1692, 1044))

In [10]:
df_dataset_tst.shape, df_signal_tst.shape, df_noise_tst.shape

((1882, 1044), (1882, 1044), (1882, 1044))

In [11]:
df_dataset_trn.to_parquet("../data/forSNR/train_test_split/dataset_trn.parquet")
df_dataset_tst.to_parquet("../data/forSNR/train_test_split/dataset_tst.parquet")
df_signal_trn.to_parquet("../data/forSNR/train_test_split/signal_trn.parquet")
df_signal_tst.to_parquet("../data/forSNR/train_test_split/signal_tst.parquet")
df_noise_trn.to_parquet("../data/forSNR/train_test_split/noise_trn.parquet")
df_noise_tst.to_parquet("../data/forSNR/train_test_split/noise_tst.parquet")